# EXP-2026-001 on Colab: GPU checks and representation gate

Thin runner. It clones code from **your fork only**, uses the RHUH-GBM data on your Drive, and calls tested scripts. No model, metric or split logic lives here.

Before running:
1. `MyDrive/RHUH-GBM` holds the RHUH-GBM release, laid out like the laptop's `data/RHUH-GBM` (your QC notebook already uses it).
2. Copy the laptop's `data/RHUH-GBM/synthesis/` folder to `MyDrive/RHUH-GBM/synthesis/`. It holds the private EXP-2026-001 manifest, frozen split, exports and policy.
3. Choose a GPU runtime (RTX PRO 6000 Blackwell, 96 GB).

**Outputs show patient IDs. Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "exp-2026-001-implementation"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
WORK = "/content/MedSegFactory"

## 1. Mount Drive, clone the fork, link the data

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
split_file = Path(DRIVE_RHUH, "synthesis", "EXP-2026-001", "split", "split.json")
assert split_file.exists(), f"Copy the laptop's data/RHUH-GBM/synthesis folder to {DRIVE_RHUH}/synthesis first"
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 2. GPU and torch build

In [ ]:
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| archs", torch.cuda.get_arch_list())
capability = torch.cuda.get_device_capability(0)
supported = capability < (12, 0) or any(arch.startswith("sm_12") for arch in torch.cuda.get_arch_list())
try:
    (torch.ones(1, device="cuda") * 2).item()
except Exception as exc:  # e.g. "no kernel image is available for execution on the device"
    supported = False
    print("CUDA kernel test failed:", exc)
print(torch.cuda.get_device_name(0), capability, "| this torch build supports the GPU:", supported)

In [ ]:
INSTALL_CU128_TORCH = False  # set True only if the cell above printed False, then restart the session
if INSTALL_CU128_TORCH:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                    "--index-url", "https://download.pytorch.org/whl/cu128", "torch", "torchvision"], check=True)
    print("Now use Runtime > Restart session, then run the notebook again from the top.")

## 3. Dependencies (versions the local tests passed with)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
                "nibabel>=5.3,<6", "SimpleITK==2.5.6", "pydicom>=3,<4", "scikit-image>=0.25"], check=True)

## 4. Verify the Drive sources match the manifest

The laptop exports were computed from the laptop's copy of RHUH-GBM. They are reused only if every pre-op/post-op T1ce and label file on Drive has the same SHA-256 as in the manifest. Reading 160 files from Drive takes a few minutes.

In [ ]:
from rhuh_synthesis.manifest import file_sha256

manifest = json.loads(Path("data/RHUH-GBM/synthesis/EXP-2026-001/manifest/case_manifest.json").read_text(encoding="utf-8"))
mismatch = []
for patient, entry in manifest["patients"].items():
    for role in ("pre", "post"):
        for kind in ("t1ce", "label"):
            record = entry[role][kind]
            path = Path(record["path"])
            if not path.exists() or file_sha256(path) != record["file_sha256"]:
                mismatch.append(f"{patient}/{role}/{kind}")
print(f"checked {4 * len(manifest['patients'])} source files; mismatches: {mismatch}")
assert not mismatch, "Drive sources differ from the manifest: stop and report before reusing the exports"

## 5. Restore the local tests and run them on this GPU machine

In [ ]:
import tarfile

snapshots = Path("data/RHUH-GBM/synthesis/EXP-2026-001/test_snapshots")
name = (snapshots / "LATEST.txt").read_text(encoding="utf-8").strip()
with tarfile.open(snapshots / name) as archive:
    try:
        archive.extractall(WORK, filter="data")
    except TypeError:  # Python without extraction filters
        archive.extractall(WORK)
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(result.stdout[-3000:])
print(result.stderr[-2000:])

## 6. Representation gate on training references (GPU)

This downloads the Stable Diffusion 1.5 VAE from `stable-diffusion-v1-5/stable-diffusion-v1-5`; its CreativeML OpenRAIL-M license was accepted on 2026-11-03. Training patients excluded by the frozen registration policy are skipped. Every path (posterior mean, seeds 11/22/33) must pass on its own. The report is written once to `synthesis/EXP-2026-001/representation_gate_v1/` on Drive.

In [ ]:
policy = json.loads(Path("data/RHUH-GBM/synthesis/EXP-2026-001/registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
command = [sys.executable, "scripts/representation_gate.py", "--allow-download", "--device", "cuda",
           "--exclude", *policy["excluded_training"]]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout[-4000:])
print(result.stderr[-3000:])

In [ ]:
import pandas as pd

report = json.loads(Path("data/RHUH-GBM/synthesis/EXP-2026-001/representation_gate_v1/report.json").read_text(encoding="utf-8"))
print("GATE", "PASS" if report["pass"] else "FAIL")
pd.DataFrame(report["paths"]).T[
    ["macro_id3_dice", "macro_id3_recall", "pooled_matched_lesions", "pooled_reference_lesions",
     "mean_spurious_id3_ml_negative", "mean_unsupported_id1_ml", "pass"]
]

## Next

Send Claude the printed GATE line and the table above (aggregate numbers only, no images). The next notebook version adds Stage 3: strict MedSegFactory checkpoint load, adapter equivalence on the real model, and one forward/backward pass on this GPU.